# Time Series Analysis

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

In [2]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from eda_utils import line_with_slider
from plotly.subplots import make_subplots

from notebooks.eda_utils import line_with_slider
from solar_predictor.data import drop_incomplete_tail

px.defaults.height = 700

In [3]:
# skopje_lp = get_location_prefix(LOCATIONS["SKOPJE"]["LAT"], LOCATIONS["SKOPJE"]["LONG"])
# raw_df = read_sarah3(skopje_lp)
# raw_df.to_csv("raw_df.csv")

raw_df = pd.read_csv("raw_df.csv", parse_dates=["datetime"]).set_index("datetime")

In [4]:
df = drop_incomplete_tail(raw_df)
daily_df = df.resample("D").mean()

daily_df.tail()

,timestamp,shortwave_radiation,direct_radiation,diffuse_radiation
datetime,,,,
2026-10-26 00:00:00+00:00,1.793015e+09,48.625000,7.812500,40.812500
2026-10-27 00:00:00+00:00,1.793102e+09,49.979167,3.041667,46.937500
2026-10-28 00:00:00+00:00,1.793188e+09,44.375000,2.333333,42.041667
2026-10-29 00:00:00+00:00,1.793274e+09,61.770833,6.604167,55.166667
2026-10-30 00:00:00+00:00,1.793361e+09,49.958333,5.208333,44.750000


# Smoothing

## Simple Moving Average (SMA) and Exponential Moving Average (EMA)

TODO: Verify shift

In [ ]:
from typing import Any

import numpy as np


def get_metrics(y_true: np.ndarray, preds: np.ndarray) -> dict[str, Any]:
    residuals = y_true - preds
    residuals = residuals.dropna()
    return {"residual_std": residuals.std(), "residual_mean": residuals.mean()}

In [ ]:
s = daily_df["shortwave_radiation"]
prev = s.shift(1)
windows = (5, 10, 21)

ma_df = pd.DataFrame({"shortwave_radiation": s})
metrics = []
for w in windows:
    ma_df[f"sma_{w}"] = prev.rolling(w).mean()
    ma_df[f"ema_{w}"] = prev.ewm(span=w, adjust=False, min_periods=w).mean()

colors = ["lightgray", *px.colors.qualitative.Plotly]
fig = make_subplots(
    rows=len(windows),
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.02,
)
for row, w in enumerate(windows, start=1):
    columns = {
        "shortwave_radiation": "shortwave_radiation",
        "sma": f"sma_{w}",
        "ema": f"ema_{w}",
    }
    for (name, column), color in zip(columns.items(), colors):
        fig.add_trace(
            go.Scatter(
                x=ma_df.index,
                y=ma_df[column],
                name=name,
                line_color=color,
                legendgroup=name,
                showlegend=row == 1,
            ),
            row=row,
            col=1,
        )
    fig.update_yaxes(title_text=f"{w} days", row=row, col=1)

fig.update_xaxes(range=[pd.to_datetime("2007-01-01"), pd.to_datetime("2008-01-01")])
fig.update_xaxes(
    rangeslider={"visible": True, "thickness": 0.05}, row=len(windows), col=1
)
fig.update_layout(
    title="Shortwave Radiation Moving Averages by Window Size",
    height=800,
    legend={
        "orientation": "h",
        "y": 1.02,
        "x": 1,
        "xanchor": "right",
        "yanchor": "bottom",
    },
)
fig.show()

display(pd.DataFrame(metrics))

Conclusion: Both methods (SMA and EMA) follow the same trend, and larger windows are smoother but react later.
The averages use only previous days, so they don't leak the predicted value.
Model validation error will show which method and window to use.

## Alpha in EMA

TODO: Maybe delete?

In [ ]:
ema_df = daily_df[["shortwave_radiation"]].copy()
prev = ema_df["shortwave_radiation"].shift(1)
ema_df["ema_0.1"] = prev.ewm(alpha=0.1, adjust=False, min_periods=1).mean()
ema_df["ema_0.2"] = prev.ewm(alpha=0.2, adjust=False, min_periods=1).mean()
ema_df["ema_0.4"] = prev.ewm(alpha=0.4, adjust=False, min_periods=1).mean()


fig = line_with_slider(
    df=ema_df,
    window=pd.DateOffset(years=1),
    color_discrete_map={"shortwave_radiation": "lightgray"},
)
fig.show()